In [3]:
import os
import sys
p = os.path.abspath('../..')
if p not in sys.path:
    sys.path.append(p)

from numcodecs import Blosc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import glob
import json
import zarr
import waveorder as wo
from recOrder.recOrder.compute.QLIPP_compute import reconstruct_QLIPP_3D, initialize_reconstructor, load_bg
from waveorder.io.reader import MicromanagerReader
from waveorder.io.writer import WaveorderWriter



In [4]:
def reconstruct_bire(position, bg_data, recon):
    
    wavelength = recon.lambda_illu*recon.n_media*1000
    
    print('Computing Birefringence...')

#     bg_data = load_bg(bg_path, height, width)

    bg_stokes = recon.Stokes_recon(bg_data)
    bg_stokes = recon.Stokes_transform(bg_stokes)
    
    stokes_data = recon.Stokes_recon(position)
    stokes_data = recon.Stokes_transform(stokes_data)

    S_image_tm = recon.Polscope_bg_correction(stokes_data, bg_stokes)
    recon_data = recon.Polarization_recon(S_image_tm)
    ret = recon_data[0] / (2 * np.pi) * wavelength
    
    return ret

## Gather Data Paths

In [5]:
data_dir = '/gpfs/CompMicro/rawdata/hummingbird/Janie/HAE 210526/63x/PTFE/pos4_z_1'
# well_name = 'Well_0'

# time_folders = glob.glob(data_dir+'*'+well_name+'*')

bg_path = '/gpfs/CompMicro/rawdata/hummingbird/Janie/HAE 210526/63x/PTFE/BG/'
bg_data = load_bg(bg_path, 2048, 2048,  cropped = False) #ROI = (225, 342, 1419, 1446),
data = MicromanagerReader(data_dir,"ometiff", extract_data=True)

In [15]:
data.slices
data.shape

(1, 6, 81, 2048, 2048)

## Initialize Reconstruction

In [6]:
## Set Reconstruction Parameters
image_dim = (2048,2048)
wavelength = 532
swing = 0.1
N_channel = 4
NA_obj = 1.2
NA_illu = 0.4
mag = 63
N_slices = 81
z_step = 0.25
pad_z = 0
pixel_size = 6.5
bg_option = 'local_fit'
n_media = 1.33

reconstructor = initialize_reconstructor(image_dim, wavelength, swing, N_channel, NA_obj, NA_illu, mag, N_slices, z_step, pad_z, 
                                         pixel_size, bg_option = bg_option, n_media = n_media, use_gpu=False, gpu_id = 0)



Initializing Reconstructor...
Finished Initializing Reconstructor (1.94 min)


## Reconstruct Batch

In [13]:
writer = WaveorderWriter('/gpfs/CompMicro/projects/HAE/210526 HAE', 'physical')
writer.create_zarr_root(f'210526_HAE_63x_PTFE_pos4_z_1_1.zarr')

Creating new zarr store at /gpfs/CompMicro/projects/HAE/210526 HAE/210526_HAE_63x_PTFE_pos4_z_1_1.zarr


In [14]:
data_shape = (1,4,81,2048,2048) #t,c,z,y,x
chunk_size = (1,1,1,2048,2048)
channels = ["Retardance","Orientation","BF","Phase3D"]

In [15]:
writer.create_position(0)
writer.init_array(data_shape, chunk_size, channels)

Creating and opening subgroup Pos_000.zarr


In [1]:
data_pos = data.get_array(0) # single position = 0, put in number of positions you want
ret_stack, ori_stack, BF_stack, phase3D = reconstruct_QLIPP_3D(data_pos[0], bg_data, reconstructor, method = "Tikhonov",
                                                                   reg_re = 1e-4, reg_im = 1e-4, rho = 1e-5, 
                                                                   lambda_re = 1e-3, lambda_im = 1e-3, itr = 20) # data_pos[0] grabs first time point
        

    

NameError: name 'data' is not defined

In [13]:
writer.write(ret_stack, t=0, c=0)
writer.write(ori_stack, t=0, c=1)
writer.write(BF_stack, t=0, c=2)
writer.write(np.transpose(phase3D, (2,0,1)), t=0, c=3)
